# With annotations

In [18]:
import os
import re
import numpy as np
import pandas as pd
import polars as pl
from towbintools.foundation.image_handling import read_tiff_file
from towbintools.foundation.image_quality import normalized_variance_measure
import matplotlib.pyplot as plt
from tifffile import imwrite
from towbintools.foundation.file_handling import add_dir_to_experiment_filemap

filemap_path = "/mnt/towbin.data/shared/spsalmon/20251014_150718_923_ZIVA_60x_443_additional_stardist_training_data/part1/analysis/report/analysis_filemap_annotated.csv"
directory_to_add = "/mnt/towbin.data/shared/spsalmon/20251014_150718_923_ZIVA_60x_443_additional_stardist_training_data/part1/raw_zstack"
filemap = pd.read_csv(filemap_path)
filemap = add_dir_to_experiment_filemap(filemap, directory_to_add, subdir_name="raw_zstack")

output_dir = "/mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/db2_part1"
output_dir = os.path.join(output_dir, "raw")
os.makedirs(output_dir, exist_ok=True)

print(filemap.columns)
# keep only rows that are not ignored
filemap = filemap[filemap['Ignore'] != True]

# for each point, keep only rows where Time > HatchTime if HatchTime is not NaN
# convert Time to numeric
filemap['Time'] = pd.to_numeric(filemap['Time'], errors='coerce')
# convert HatchTime to numeric
filemap['HatchTime'] = pd.to_numeric(filemap['HatchTime'], errors='coerce')

filemap = filemap[(filemap['HatchTime'].isna()) | (filemap['Time'] > filemap['HatchTime'])]
filemap.head(77)

interesting_stacks = filemap['raw_zstack'].tolist()
# remove empty strings
interesting_stacks = [s for s in interesting_stacks if s != ""]

print(f"Number of interesting stacks: {len(interesting_stacks)}")

number_of_stacks = 100
planes_per_stack = 2
np.random.shuffle(interesting_stacks)
picked_stacks = interesting_stacks[:number_of_stacks]

Index(['Time', 'Point', 'raw', 'ExperimentTime', 'analysis/ch1_seg',
       'ch1_seg_area', 'placeholder_worm_type', 'ch1_seg_area_at_HatchTime',
       'ch1_seg_area_at_M1', 'ch1_seg_area_at_M2', 'ch1_seg_area_at_M3',
       'ch1_seg_area_at_M4', 'HatchTime', 'M1', 'M2', 'M3', 'M4', 'Ignore',
       'raw_zstack'],
      dtype='object')
Number of interesting stacks: 416


In [20]:
for stack in picked_stacks:
    test_stack = read_tiff_file(stack, channels_to_keep=[1])

    scores = []
    for i in range(test_stack.shape[0]):
        slice = test_stack[i]
        score = normalized_variance_measure(slice)
        scores.append(score)

    best_plane = np.argmax(scores)
    planes_with_data = test_stack[best_plane-2:]

    try:
        random_planes = np.random.choice(
            np.arange(planes_with_data.shape[0]),
            size=planes_per_stack,
            replace=False
        )
    except ValueError as e:
        print("Not enough planes to choose from:", e)
        continue

    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        tile_size = 512
        # take N random crops of size tile_size x tile_size
        num_crops = 50
        crops = []
        for _ in range(num_crops):
            x = np.random.randint(0, random_plane.shape[0] - tile_size)
            y = np.random.randint(0, random_plane.shape[1] - tile_size)
            crop = random_plane[x:x+tile_size, y:y+tile_size]
            crops.append(crop)

        crops = np.array(crops)
        scores = []
        for crop in crops:
            score = normalized_variance_measure(crop)
            scores.append(score)

        # remove crops with too low scores
        scores = np.array(scores)
        print(np.median(scores), np.mean(scores), np.std(scores))
        valid_indices = np.where(scores > 1.)[0]
        scores = scores[valid_indices]
        crops = crops[valid_indices]

        try:
            # randomly pick a crop among the top 10 crops
            top_indices = np.argsort(scores)[-10:]
            chosen_index = np.random.choice(top_indices)
            best_crop = crops[chosen_index]
            random_hex_hash = os.urandom(4).hex()
            imwrite(os.path.join(output_dir, f"crop_{random_hex_hash}.tiff"), best_crop, compression="zlib")
            
        except Exception as e:
            print("Not enough valid crops to choose from:", e)

0.683307543650477 0.6787534786952686 0.024041337089829348
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.7318693940261753 0.7385616824514527 0.07809755790746695
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.8915528622479951 0.8920120036059883 0.1336353795836347
0.7350227262915634 0.7840962590398989 0.09858730243599681
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6611799269269065 0.6605471510818752 0.024064445363017026
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6692420508866717 0.6682869180173757 0.018191918417536614
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.6564209239733159 0.6564870488729142 0.017251085070149174
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.7686582808540425 0.7827514069604387 0.11964736711153531
0.7214120832

# No annotations

In [1]:
import os
import re
import numpy as np
from towbintools.foundation.image_handling import read_tiff_file
from towbintools.foundation.image_quality import normalized_variance_measure
import matplotlib.pyplot as plt
from tifffile import imwrite
experiment_dir = "/mnt/towbin.data/shared/spsalmon/20251014_150718_923_ZIVA_60x_443_additional_stardist_training_data/part2/"

output_dir = os.path.join(experiment_dir, "db2_part2", "raw")
os.makedirs(output_dir, exist_ok=True)

zstack_dir_path = os.path.join(experiment_dir, "raw_zstack")
all_zstack_files = [
    os.path.join(zstack_dir_path, f)
    for f in os.listdir(zstack_dir_path)
    if f.endswith(".tiff")
]

number_of_stacks = 100
planes_per_stack = 2
np.random.shuffle(all_zstack_files)
picked_stacks = all_zstack_files[:number_of_stacks]

In [3]:
for stack in picked_stacks:
    test_stack = read_tiff_file(stack, channels_to_keep=[1])

    scores = []
    for i in range(test_stack.shape[0]):
        slice = test_stack[i]
        score = normalized_variance_measure(slice)
        scores.append(score)

    best_plane = np.argmax(scores)
    planes_with_data = test_stack[best_plane-2:]

    random_planes = np.random.choice(
        np.arange(planes_with_data.shape[0]),
        size=planes_per_stack,
        replace=False
    )

    for random_plane in random_planes:
        random_plane = planes_with_data[random_plane]
        tile_size = 512
        # take N random crops of size tile_size x tile_size
        num_crops = 50
        crops = []
        for _ in range(num_crops):
            x = np.random.randint(0, random_plane.shape[0] - tile_size)
            y = np.random.randint(0, random_plane.shape[1] - tile_size)
            crop = random_plane[x:x+tile_size, y:y+tile_size]
            crops.append(crop)

        crops = np.array(crops)
        scores = []
        for crop in crops:
            score = normalized_variance_measure(crop)
            scores.append(score)

        # remove crops with too low scores
        scores = np.array(scores)
        print(np.median(scores), np.mean(scores), np.std(scores))
        valid_indices = np.where(scores > 1.)[0]
        scores = scores[valid_indices]
        crops = crops[valid_indices]

        try:
            # randomly pick a crop among the top 10 crops
            top_indices = np.argsort(scores)[-10:]
            chosen_index = np.random.choice(top_indices)
            best_crop = crops[chosen_index]
            random_hex_hash = os.urandom(4).hex()
            imwrite(os.path.join(output_dir, f"crop_{random_hex_hash}.tiff"), best_crop, compression="zlib")
            
        except Exception as e:
            print("Not enough valid crops to choose from:", e)



1.2598700167877945 1.2445728950242556 0.3598805110766617
1.2770812156440043 1.304003977964121 0.3934634071441232
0.9410696762338038 0.9493837208016571 0.103320324772254
1.16595830748739 1.1838945017302327 0.14277856588447563
0.7994232913320614 0.7974375079783634 0.04546376976229037
Not enough valid crops to choose from: 'a' cannot be empty unless no samples are taken
0.988623540779338 1.0114753837351267 0.11910241521782088
1.768510298597643 1.7840038074762448 0.2914783772686185
1.2865451622686994 1.355563370677612 0.29558945332231107
1.0279868180340905 1.0075392969808177 0.1063955313780513
0.9119272289763496 0.8962122415772936 0.09079032014373431
1.0286737133590242 1.0440811527200626 0.20044606505429785
0.8749647070829036 0.8900525749261894 0.16297243149948584
1.0940329411100636 1.0710554900080949 0.1613039300513952
1.013326107971788 1.0207592736135833 0.10001620350487725
1.0268040030715282 1.0361757023247333 0.1211461945863783
1.0017684057299134 1.003219013149294 0.11753492436378081
1